# LangChain 소개와 첫 모델 호출 — 주피터 따라 하기
이 노트북은 **완성된 코드**입니다. 위에서부터 한 셀씩 `Shift + Enter`로 실행하면서, 셀마다 **무엇을 보냈고(요청) 무엇을 받았는지(출력)** 확인합니다.

- 같은 가상 인계 메모를 **OpenAI SDK**와 **LangChain `ChatOpenAI`** 로 각각 보내 요청 형식과 답을 꺼내는 위치를 비교합니다.
- 모든 셀은 ▶ 제공 코드입니다. 빈칸이나 채점 셀은 없습니다.
- 직접 구현하는 과제는 별도 런박스에서 합니다. 같은 폴더의 `바이브코딩_인계지시문/실습_인계지시문_바이브코딩.html`을 여세요.

이 노트북은 **독립 Runbox용**입니다. 다른 노트북을 열지 않아도 필요한 코드가 모두 들어 있습니다.

### 0. 실습 환경 확인
Python 버전이 **3.12.x**(3.12로 시작)로 나오면 정상입니다.

In [2]:
%pip install langchain langchain-core langchain-openai langgraph

  Using cached langgraph-1.2.24-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.14.4-py3-none-any.whl.metadata (143 kB)
  Using cached certifi-2026.11.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.48-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_protocol-0.0.22-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.16.2-py3-none-any.whl.metadata (40 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached langgraph_checkpoint-4.3.2-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached langgraph_sdk-0.4.14-py3-none-any.whl.metadata (5.1 kB)
  Using cached xxhash-4.1.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
  Usin

Using cached jsonpointer-3.12-py3-none-any.whl (10 kB)
Using cached langchain_protocol-0.0.22-py3-none-any.whl (7.6 kB)
Using cached openai-3.59.0-py3-none-any.whl (2.5 MB)
Using cached jiter-0.19.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (338 kB)
Using cached orjson-3.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (130 kB)
Using cached ormsgpack-1.13.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (273 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.3-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (257 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached tenacity-9.2.1-py3-none-any.whl (32 kB)
Using cached tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl (1.2 MB)
Using cached truststore-0.10.5-py3-none-any.whl (19 kB)
Using cached typing_inspection-0.4.5-py3-none-any.whl (14 

In [3]:
# ▶ 제공 코드 — 실습 환경(커널) 확인
import sys
from importlib.metadata import version

print("Python", sys.version.split()[0], "(기준 3.12.x)")
for name in ["langchain", "langchain-core", "langchain-openai", "langgraph"]:
    print(f"{name:18s}", version(name))

Python 3.12.9 (기준 3.12.x)
langchain          1.5.2
langchain-core     1.8.5
langchain-openai   1.8.5
langgraph          1.2.24


### 1. 설정 셀 — 모델 연결
안내받은 **수업용 API 키**(`API_KEY`)와 **채팅 모델 주소**(`BASE_URL`)를 따옴표 안에 직접 입력하세요. 모델은 `gpt-6-luna`로 정해져 있어 바꾸지 않습니다. 이 실습은 채팅 모델만 쓰므로 임베딩 설정은 없습니다.
- 두 값 중 하나라도 자리표시자로 남아 있으면 이 셀이 모델을 호출하기 전에 멈추고 무엇을 입력할지 알려 줍니다.
- `.env` 파일은 사용하지 않습니다.
- 이 셀은 AI 채팅창이나 AI 코딩 도우미에 **붙여넣지 마세요**. 노트북을 제출·공유하기 전에는 키를 다시 `"여기에_수업용_API_키_입력"`으로 되돌립니다.

In [4]:
# ▶ 제공 코드 — 설정 셀 (따옴표 안의 값만 바꾸세요)
API_KEY = ""
BASE_URL = ""   # 안내받은 채팅 모델 주소(https://...)
MODEL_NAME = "gpt-6-luna"                  # 수업에서 쓰는 채팅·도구 호출 모델(확정값, 바꾸지 않음)

missing = [name for name, value in (("API_KEY", API_KEY), ("BASE_URL", BASE_URL)) if value.startswith("여기에")]
if missing:
    raise ValueError(", ".join(missing) + "에 안내받은 값을 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL)
print("모델 준비 완료:", MODEL_NAME)

모델 준비 완료: gpt-6-luna


### 2. 보낼 메모와 지시 준비
두 방식에 **똑같은 메모와 지시**를 보냅니다. 아직 모델을 호출하지 않았습니다.

In [5]:
# ▶ 제공 코드 — 가상 업무 메모(실제 공정 정보가 아닙니다)와 역할 지시
memo = """A라인 온도 센서 교체 완료.
14시 재가동 예정.
재가동 후 30분 동안 온도 로그 확인 필요."""

SYSTEM_TEXT = "당신은 제조 현장의 교대 인계 도우미입니다. 다음 근무자가 읽기 쉽게 메모를 짧게 정리하세요."
print("지시:", SYSTEM_TEXT)
print("메모:", memo)

지시: 당신은 제조 현장의 교대 인계 도우미입니다. 다음 근무자가 읽기 쉽게 메모를 짧게 정리하세요.
메모: A라인 온도 센서 교체 완료.
14시 재가동 예정.
재가동 후 30분 동안 온도 로그 확인 필요.


### 3. SDK로 직접 호출하기
모델 회사가 제공하는 기본 도구(SDK)는 역할(`role`)과 내용(`content`)을 담은 **딕셔너리 목록**을 보냅니다. 답은 `choices[0].message.content`에 있습니다. 이 셀을 실행할 때마다 메모를 모델에 다시 보냅니다.

In [6]:
# ▶ 제공 코드 — OpenAI SDK 호출
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
sdk_messages = [
    {"role": "system", "content": SYSTEM_TEXT},
    {"role": "user", "content": memo},
]
response = client.chat.completions.create(model=MODEL_NAME, messages=sdk_messages)

print("보낸 요청:", [(m["role"], m["content"][:20]) for m in sdk_messages])
print("답을 꺼낸 위치: response.choices[0].message.content")
print(response.choices[0].message.content)

보낸 요청: [('system', '당신은 제조 현장의 교대 인계 도우미'), ('user', 'A라인 온도 센서 교체 완료.\n14시')]
답을 꺼낸 위치: response.choices[0].message.content
- A라인 온도 센서 교체 완료
- 14시 재가동 예정
- 재가동 후 30분간 온도 로그 확인 필요


### 4. LangChain으로 같은 메모 보내기
LangChain은 역할마다 **메시지 객체**(`SystemMessage`, `HumanMessage`)를 만들어 `llm.invoke()`에 넘깁니다. 결과는 **AIMessage**이고 답변 글자는 `.content`에 있습니다.

In [7]:
# ▶ 제공 코드 — ChatOpenAI 호출
from langchain.messages import SystemMessage, HumanMessage

messages = [SystemMessage(SYSTEM_TEXT), HumanMessage(memo)]
answer = llm.invoke(messages)

print("보낸 요청:", [(m.type, m.content[:20]) for m in messages])
print("결과의 종류:", type(answer).__name__)
print("답을 꺼낸 위치: answer.content")
print(answer.content)

보낸 요청: [('system', '당신은 제조 현장의 교대 인계 도우미'), ('human', 'A라인 온도 센서 교체 완료.\n14시')]
결과의 종류: AIMessage
답을 꺼낸 위치: answer.content
- **A라인 온도 센서 교체 완료**
- **14시 재가동 예정**
- 재가동 후 **30분간 온도 로그 확인 필요**


### 5. 이미 받은 응답 확인하기
- `answer.content` → 답변 내용
- `answer.usage_metadata` → 사용량 정보(서버가 제공할 때). 비용은 각 요청에서 쓴 입력·출력 토큰으로 계산합니다.

아래 셀은 이미 받은 `answer`를 읽기만 하므로 새 요청을 보내지 않습니다.

In [8]:
# ▶ 제공 코드 — 이미 받은 응답에서 내용과 사용량 꺼내기
print("답변 내용 :", answer.content[:40], "...")
print("사용량 정보:", answer.usage_metadata)   # 수업용 서버에 따라 None으로 나올 수도 있습니다

답변 내용 : - **A라인 온도 센서 교체 완료**
- **14시 재가동 예정**
- ...
사용량 정보: {'input_tokens': 73, 'output_tokens': 89, 'total_tokens': 162, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 44}}


### 6. 두 방식 나란히 보기
| 비교 | OpenAI SDK | LangChain |
|---|---|---|
| 모델 준비 | `OpenAI(...)` + 호출마다 `model=` | `ChatOpenAI(model=...)` 한 번 |
| 보낼 메시지 | `{"role": "system", ...}` 딕셔너리 | `SystemMessage(...)`, `HumanMessage(...)` 객체 |
| 보내기 | `client.chat.completions.create(...)` | `llm.invoke(...)` |
| 답 꺼내기 | `response.choices[0].message.content` | `answer.content` |

In [9]:
# ▶ 제공 코드 — 두 답을 나란히 출력 (새 요청 없음)
print("[SDK]      ", response.choices[0].message.content)
print("[LangChain]", answer.content)
print("같은 메모·같은 지시를 보냈어도 문장은 매번 조금 다를 수 있습니다.")

[SDK]       - A라인 온도 센서 교체 완료
- 14시 재가동 예정
- 재가동 후 30분간 온도 로그 확인 필요
[LangChain] - **A라인 온도 센서 교체 완료**
- **14시 재가동 예정**
- 재가동 후 **30분간 온도 로그 확인 필요**
같은 메모·같은 지시를 보냈어도 문장은 매번 조금 다를 수 있습니다.


### 7. 메모만 바꿔 다시 보내기
업무 규칙(지시)은 그대로 두고, 교대마다 바뀌는 **메모만** 사용자 메시지로 바꿉니다. 재가동 예정 시각을 14시에서 15시로 바꾸면 답의 시각도 바뀌는지 확인하세요. 이 셀을 실행하면 모델에 새 요청을 한 번 보냅니다.

In [10]:
# ▶ 제공 코드 — 메모만 바꾼 요청
memo_changed = memo.replace("14시", "15시")
answer_changed = llm.invoke([SystemMessage(SYSTEM_TEXT), HumanMessage(memo_changed)])

print("바뀐 메모:", memo_changed.splitlines()[1])
print(answer_changed.content)
print("답에 15시 포함:", "15시" in answer_changed.content, "/ 14시 남음:", "14시" in answer_changed.content)

바뀐 메모: 15시 재가동 예정.
- A라인 온도 센서 교체 완료
- 15시 재가동 예정
- 재가동 후 30분간 온도 로그 확인 필요
답에 15시 포함: True / 14시 남음: False


### 정리
| 확인한 것 | 어디서 보았나 |
|---|---|
| SDK는 딕셔너리 목록, LangChain은 메시지 객체로 요청 | 단계 3·4의 ‘보낸 요청’ 줄 |
| 답의 위치: `choices[0].message.content` / `answer.content` | 단계 3·4·6 |
| 지시는 그대로, 메모만 바꾸면 답의 시각이 바뀜 | 단계 7 |

**이어서 할 일** — 런박스 과제에서 지금의 짧은 지시를 **완료·예정·확인 세 줄 인계 지시문**으로 직접 바꿉니다. 안내: `바이브코딩_인계지시문/실습_인계지시문_바이브코딩.html`